In [ ]:
# The original dataset loading script works more reliably with older versions
# of the Hugging Face libraries, so we install compatible versions here.
# The uninstall command is commented out because it is only needed if Colab
# already has an incompatible version installed.
# !pip uninstall -y datasets
!pip install -q "datasets<4.0.0" "huggingface_hub<1.0.0"

# Import the libraries
import datasets, huggingface_hub
from datasets import load_dataset
from google.colab import files
import pandas as pd
import re

# Download the NLTK resources needed for the LDA preprocessing pipeline.
import nltk
# punkt and punkt_tab are used for tokenization.
# punkt_tab is required by newer NLTK versions in some Colab environments
nltk.download("punkt")
nltk.download("punkt_tab")
# stopwords provides the English stopword list
nltk.download("stopwords")
# wordnet and omw-1.4 are used for lemmatization
nltk.download("wordnet")
nltk.download("omw-1.4")
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Print versions to document the environment
print("datasets==", datasets.__version__)
print("huggingface-hub==", huggingface_hub.__version__)
print("pandas==", pd.__version__)
print("nltk==", nltk.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
transformers 5.12.1 requires huggingface-hub<2.0,>=1.5.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have huggingface-hub 0.36.2 which is incompatible.


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


3.6.0
1.20.1


In [ ]:
# The Webis-TLDR-17 dataset loader downloads the original archive
# corpus-webis-tldr-17.zip, which is about 3.1 GB.
# Because of this, even loading a small subset may require access
# to the full archive first

# We run this step in Google Colab because the temporary storage is provided
# by Google's servers, so the large archive does not need to be stored locally

# Load the first 1,000 examples from the training split.
# This small subset is used first to check that the dataset loads correctly
# and to inspect the available columns before working with a larger sample
ds = load_dataset(
    "webis/tldr-17",
    split="train[:1000]",
    trust_remote_code=True
)

df = ds.to_pandas()
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dataset_infos.json: 0.00B [00:00, ?B/s]

data/corpus-webis-tldr-17.zip:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3848330 [00:00<?, ? examples/s]

,author,body,normalizedBody,subreddit,subreddit_id,id,content,summary
0,raysofdarkmatter,I think it should be fixed on either UTC stand...,I think it should be fixed on either UTC stand...,math,t5_2qh0n,c69al3r,I think it should be fixed on either UTC stand...,Shifting seasonal time is no longer worth it.
1,Stork13,Art is about the hardest thing to categorize i...,Art is about the hardest thing to categorize i...,funny,t5_2qh33,c6a9nxd,Art is about the hardest thing to categorize i...,Personal opinions 'n shit.
2,Cloud_dreamer,Ask me what I think about the Wall Street Jour...,Ask me what I think about the Wall Street Jour...,Borderlands,t5_2r8cd,c6acx4l,Ask me what I think about the Wall Street Jour...,insults and slack ass insight. \n Wall Street ...
3,NightlyReaper,"In Mechwarrior Online, I have begun to use a m...","In Mechwarrior Online, I have begun to use a m...",gamingpc,t5_2sq2y,c8onqew,"In Mechwarrior Online, I have begun to use a m...","Yes, Joysticks in modern games have apparently..."
4,NuffZetPand0ra,"You are talking about the Charsi imbue, right?...","You are talking about the Charsi imbue, right?...",Diablo,t5_2qore,c6acxvc,"You are talking about the Charsi imbue, right?...",Class only items dropped from high-lvl monsters.


In [ ]:
# Load a larger subset of the training split.
# We use 100,000 examples instead of the full dataset to make the analysis
# faster and more manageable in Colab, while still having enough data
# to find subreddits with at least 1,000 examples
ds = load_dataset(
    "webis/tldr-17",
    split="train[:100000]",
    trust_remote_code=True
)

df = ds.to_pandas()
# Check the number of rows and columns in the loaded subset
df.shape

data/corpus-webis-tldr-17.zip:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3848330 [00:00<?, ? examples/s]

(100000, 8)

In [ ]:
# Count how many content entries are available for each subreddit.
subreddit_stats = (
    df.groupby("subreddit")
    .agg(
        n_contents=("content", "count")
    )
    .sort_values("n_contents", ascending=False)
    .reset_index()
)

# Display the 30 subreddits with the most examples in the loaded subset
subreddit_stats.head(30)

,subreddit,n_contents
0,AskReddit,23526
1,leagueoflegends,2374
2,AdviceAnimals,1874
3,funny,1787
4,gaming,1623
5,pics,1556
6,politics,1519
7,atheism,1430
8,explainlikeimfive,1146
9,todayilearned,1011


In [ ]:
# Keep only subreddits that have at least 1,000 content entries
eligible_subreddits = subreddit_stats[subreddit_stats["n_contents"] >= 1000]
eligible_subreddits

,subreddit,n_contents
0,AskReddit,23526
1,leagueoflegends,2374
2,AdviceAnimals,1874
3,funny,1787
4,gaming,1623
5,pics,1556
6,politics,1519
7,atheism,1430
8,explainlikeimfive,1146
9,todayilearned,1011


In [ ]:
# Select the four subreddits used in the final sample.
SELECTED_SUBREDDITS = [
    "politics",
    "gaming",
    "todayilearned",
    "explainlikeimfive",
]

# Show the counts only for the selected subreddits.
# This confirms that each selected subreddit has at least 1,000 examples.
selected_stats = subreddit_stats[
    subreddit_stats["subreddit"].isin(SELECTED_SUBREDDITS)
].sort_values("subreddit")

selected_stats

,subreddit,n_contents
8,explainlikeimfive,1146
4,gaming,1623
6,politics,1519
9,todayilearned,1011


In [ ]:
# Create the final working sample
# First, keep only rows from the four selected subreddits.
sample_df = (
    df[df["subreddit"].isin(SELECTED_SUBREDDITS)]

    # Remove rows where either the post content or the summary is missing
    .dropna(subset=["content", "summary"])

    # Remove rows where content or summary is an empty string.
    .query("content.str.strip() != '' and summary.str.strip() != ''", engine="python")

    # Randomly sample 1,000 examples from each selected subreddit
    # random_state makes the sampling reproducible
    .groupby("subreddit", group_keys=False)
    .apply(lambda x: x.sample(n=1000, random_state=42))

    # Reset the row index after sampling
    .reset_index(drop=True)
)

# Check that the final dataset contains exactly 1,000 examples per subreddit
sample_df["subreddit"].value_counts()

/tmp/ipykernel_8353/1723144375.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=1000, random_state=42))


,count
subreddit,
explainlikeimfive,1000
gaming,1000
politics,1000
todayilearned,1000


In [ ]:
sample_df.head()

,author,body,normalizedBody,subreddit,subreddit_id,id,content,summary
0,Aintsley,I don't know where you access your information...,I don't know where you access your information...,explainlikeimfive,t5_2sokd,ckxy36n,I don't know where you access your information...,about 4% for the (65) age group develop ... \n...
1,Jiveturkeey,Some Americans are obsessed with guns. Many of...,Some Americans are obsessed with guns. Many of...,explainlikeimfive,t5_2sokd,ce72xym,Some Americans are obsessed with guns. Many of...,Guns are part of American culture due to our r...
2,JimMarch,The reason Solzhenitsyn was thrown out of the ...,The reason Solzhenitsyn was thrown out of the ...,explainlikeimfive,t5_2sokd,cl9u78u,The reason Solzhenitsyn was thrown out of the ...,I have no use for communist apologetics.
3,MyNameIsRay,"There are some exceptions. Trader Joe's wine, ...","There are some exceptions. Trader Joe's wine, ...",explainlikeimfive,t5_2sokd,coodnvc,"There are some exceptions. Trader Joe's wine, ...","cheap can be the way it's supposed to be, if y..."
4,Lorikeet,Interesting.. But how enforced is this? About ...,Interesting.. But how enforced is this? About ...,explainlikeimfive,t5_2sokd,ce8uqrb,Interesting.. But how enforced is this? About ...,"don't fly United, ever."


In [ ]:
# Save the final sampled dataset as a CSV file
sample_df.to_csv("selected_4_subreddits_1000_each_with_lengths.csv", index=False)

# Download the CSV file from Google Colab
files.download("selected_4_subreddits_1000_each_with_lengths.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# If sample_df is already in memory, you can skip this line
# Otherwise load the saved file:
sample_df = pd.read_csv("selected_4_subreddits_1000_each_with_lengths.csv")

def normal_clean(text):
    if pd.isna(text):
        return ""

    text = str(text)

    # Remove URLs and links
    text = re.sub(r"http\S+|www\.\S+", " ", text)

    # Remove HTML tags if present
    text = re.sub(r"<.*?>", " ", text)

    # Remove Reddit markdown links: [text](url) -> text
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)

    # Remove extra whitespace, tabs, new lines
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [ ]:
# Apply the normal cleaning function to the original post content
# The cleaned version is stored in a new column, while the original column is kept unchanged
sample_df["content_clean"] = sample_df["content"].apply(normal_clean)

# Apply the same cleaning function to the TL;DR summaries
sample_df["summary_clean"] = sample_df["summary"].apply(normal_clean)

# Compare the original and cleaned text columns
sample_df[["content", "content_clean", "summary", "summary_clean"]].head()

,content,content_clean,summary,summary_clean
0,I don't know where you access your information...,I don't know where you access your information...,about 4% for the (65) age group develop ... \n...,about 4% for the (65) age group develop ... gr...
1,Some Americans are obsessed with guns. Many of...,Some Americans are obsessed with guns. Many of...,Guns are part of American culture due to our r...,Guns are part of American culture due to our r...
2,The reason Solzhenitsyn was thrown out of the ...,The reason Solzhenitsyn was thrown out of the ...,I have no use for communist apologetics.,I have no use for communist apologetics.
3,"There are some exceptions. Trader Joe's wine, ...","There are some exceptions. Trader Joe's wine, ...","cheap can be the way it's supposed to be, if y...","cheap can be the way it's supposed to be, if y..."
4,Interesting.. But how enforced is this? About ...,Interesting.. But how enforced is this? About ...,"don't fly United, ever.","don't fly United, ever."


In [ ]:
# Stopwords are frequent function words such as "the", "and", "is",
# which usually do not carry much topic information for LDA
stop_words = set(stopwords.words("english"))

# Initialize the WordNet lemmatizer
# Lemmatization reduces words to their base form, e.g. "running" -> "running"/"run"
lemmatizer = WordNetLemmatizer()

def lda_preprocess(text):
    # Replace missing values with an empty string
    if pd.isna(text):
        return ""

    # Convert text to lowercase to avoid treating "Game" and "game" as different words
    text = str(text).lower()

    # Tokenize the text into individual word-like units
    tokens = word_tokenize(text)

    processed_tokens = []

    for token in tokens:
        # Keep only alphabetic characters.
        # This removes punctuation, numbers, and special symbols
        token = re.sub(r"[^a-z]", "", token)

        # Skip empty tokens that may result from cleaning.
        if token == "":
            continue

        # Remove stopwords.
        if token in stop_words:
            continue

        # Remove very short tokens because they are often not meaningful for topics
        if len(token) <= 2:
            continue

        # Lemmatize the token to reduce inflected forms to a common base form
        lemma = lemmatizer.lemmatize(token)

        processed_tokens.append(lemma)

    # Return the processed tokens as one whitespace-separated string
    # This format can later be used for vectorization and LDA
    return " ".join(processed_tokens)

In [ ]:
# Apply the LDA-specific preprocessing function to the cleaned post content
# The result is stored in a new column and the previous columns are kept unchanged
sample_df["content_lda_preprocessed"] = sample_df["content_clean"].apply(lda_preprocess)

# Apply the same LDA-specific preprocessing to the cleaned TL;DR summaries
sample_df["summary_lda_preprocessed"] = sample_df["summary_clean"].apply(lda_preprocess)

# Compare the lightly cleaned text with the LDA-preprocessed version
sample_df[
    [
        "content_clean",
        "content_lda_preprocessed",
        "summary_clean",
        "summary_lda_preprocessed"
    ]
].head()

,content_clean,content_lda_preprocessed,summary_clean,summary_lda_preprocessed
0,I don't know where you access your information...,know access information claiming male develop ...,about 4% for the (65) age group develop ... gr...,age group develop gradually range
1,Some Americans are obsessed with guns. Many of...,american obsessed gun many member nra make lot...,Guns are part of American culture due to our r...,gun part american culture due revolutionary be...
2,The reason Solzhenitsyn was thrown out of the ...,reason solzhenitsyn thrown ussr breshnev era t...,I have no use for communist apologetics.,use communist apologetics
3,"There are some exceptions. Trader Joe's wine, ...",exception trader joe wine two buck chuck calle...,"cheap can be the way it's supposed to be, if y...",cheap way supposed know look expensive guarant...
4,Interesting.. But how enforced is this? About ...,interesting enforced two year ago told flight ...,"don't fly United, ever.",fly united ever


In [ ]:
# Save the final dataset with all original columns and the added preprocessing columns
# This file includes:
# - content_len and summary_len
# - content_clean and summary_clean
# - content_lda_preprocessed and summary_lda_preprocessed

sample_df.to_csv("selected_4_subreddits_1000_each_clean_lda.csv", index=False)
files.download("selected_4_subreddits_1000_each_clean_lda.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>